# V2_02 — Stage 1: No-Charge Ablation

Train XGBoost, CatBoost, and LightGBM on full data **without `Avg_Sbmtd_Chrg`**.

This quantifies the model's honest predictive power — how well can we predict the allowed amount
without already knowing the billed charge? V1 RF had 61.8% feature importance on `Avg_Sbmtd_Chrg`.

**Runtime:** T4 GPU + High-RAM | ~3-5 hrs | ~6-10 CU

**Prerequisite:** Gold parquets uploaded to `My Drive/AllowanceMap/V2/gold/`

In [ ]:
# ── Cell 1: Environment Setup ──────────────────────────────────────────────
import os, subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'mlflow>=2.12', 'catboost>=1.2', 'lightgbm>=4.3', 'databricks-sdk>=0.20'])

from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AllowanceMap/V2'
GOLD_DIR   = f'{DRIVE_ROOT}/gold'
ARTIFACTS  = f'{DRIVE_ROOT}/v2_artifacts'
os.makedirs(f'{ARTIFACTS}/models', exist_ok=True)
os.makedirs(f'{ARTIFACTS}/plots', exist_ok=True)

os.environ['DATABRICKS_HOST']  = 'https://dbc-d709cbb6-fe84.cloud.databricks.com'
os.environ['DATABRICKS_TOKEN'] = userdata.get('DATABRICKS_TOKEN')

import mlflow, requests
mlflow.set_tracking_uri('databricks')
resp = requests.get(
    f"{os.environ['DATABRICKS_HOST']}/api/2.0/preview/scim/v2/Me",
    headers={'Authorization': f"Bearer {os.environ['DATABRICKS_TOKEN']}"},
    timeout=10,
)
resp.raise_for_status()
USER_HOME = f"/Users/{resp.json()['userName']}"
mlflow.set_experiment(f'{USER_HOME}/medicare_models')
print(f'MLflow: {USER_HOME}/medicare_models')

In [ ]:
# ── Cell 2: Constants — NO-CHARGE feature list ─────────────────────────────
import glob, gc, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor, Pool

TARGET = 'Avg_Mdcr_Alowd_Amt'

FEATURES_FULL = [
    'Rndrng_Prvdr_Type_idx', 'Rndrng_Prvdr_State_Abrvtn_idx',
    'HCPCS_Cd_idx', 'hcpcs_bucket', 'place_of_srvc_flag',
    'Bene_Avg_Risk_Scre', 'log_srvcs', 'log_benes',
    'Avg_Sbmtd_Chrg', 'srvcs_per_bene',
    'specialty_bucket', 'pos_bucket', 'hcpcs_target_enc',
]

# *** ABLATION: remove Avg_Sbmtd_Chrg ***
FEATURES = [f for f in FEATURES_FULL if f != 'Avg_Sbmtd_Chrg']
print(f'Ablation features ({len(FEATURES)}): {FEATURES}')
print(f'Removed: Avg_Sbmtd_Chrg')

CAT_FEATURE_NAMES = [
    'Rndrng_Prvdr_Type_idx', 'Rndrng_Prvdr_State_Abrvtn_idx',
    'HCPCS_Cd_idx', 'hcpcs_bucket', 'place_of_srvc_flag',
    'specialty_bucket', 'pos_bucket',
]

def get_cat_indices(features):
    return [i for i, f in enumerate(features) if f in CAT_FEATURE_NAMES]

def make_pool(X_df, y, features, cat_indices):
    """Create CatBoost Pool from DataFrame with proper categorical handling."""
    X = X_df[features].copy()
    for idx in cat_indices:
        col = features[idx]
        X[col] = X[col].astype(int)
    return Pool(X, label=y, cat_features=cat_indices, feature_names=features)

def log_metrics(y_true_log, y_pred_log, prefix=''):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mlflow.log_metrics({f'{prefix}mae': mae, f'{prefix}rmse': rmse, f'{prefix}r2': r2})
    print(f'  {prefix}MAE=${mae:.2f}  RMSE=${rmse:.2f}  R²={r2:.4f}')
    return mae, rmse, r2

In [ ]:
# ── Cell 3: Load Full Dataset → save splits to local SSD ──────────────────
print('Loading all gold parquets...')
t0 = time.time()

load_cols = list(dict.fromkeys(FEATURES + [TARGET]))   # no 'year' needed
parts = []
for f in sorted(glob.glob(os.path.join(GOLD_DIR, '*.parquet'))):
    avail = set(pq.read_schema(f).names)
    cols = [c for c in load_cols if c in avail]
    df = pd.read_parquet(f, columns=cols).dropna(subset=FEATURES + [TARGET])
    parts.append(df)

df_all = pd.concat(parts, ignore_index=True)
del parts; gc.collect()
print(f'Loaded {len(df_all):,} rows in {time.time()-t0:.0f}s')

# log1p target
df_all[TARGET] = np.log1p(df_all[TARGET].astype('float64'))

# Split on indices, save to local SSD
idx_train, idx_test = train_test_split(np.arange(len(df_all)), test_size=0.2, random_state=42)
n_train = len(idx_train)
n_test  = len(idx_test)

df_all.iloc[idx_train].to_parquet('/content/train_split.parquet', index=False)
df_all.iloc[idx_test].to_parquet('/content/test_split.parquet', index=False)
del df_all; gc.collect()

print(f'Train: {n_train:,} | Test: {n_test:,}')
print('Saved splits to /content/train_split.parquet & /content/test_split.parquet')

In [ ]:
# ── Cell 4: XGBoost No-Charge ──────────────────────────────────────────────
import xgboost as xgb

# Reload splits from local SSD
df_train = pd.read_parquet('/content/train_split.parquet')
df_test  = pd.read_parquet('/content/test_split.parquet')
X_train = df_train[FEATURES].astype('float32').values
y_train = df_train[TARGET].values
X_test  = df_test[FEATURES].astype('float32').values
y_test  = df_test[TARGET].values
del df_train, df_test; gc.collect()

XGB_PARAMS = {
    'objective': 'reg:squarederror', 'learning_rate': 0.05, 'max_depth': 6,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'tree_method': 'hist',
    'device': 'cuda', 'seed': 42, 'max_bin': 256,
}

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURES)
del X_train; gc.collect()

print('Training XGBoost (no charge)...')
t0 = time.time()
xgb_booster = xgb.train(
    XGB_PARAMS, dtrain, num_boost_round=1000,
    evals=[(dtest, 'val')], early_stopping_rounds=50, verbose_eval=100,
)
xgb_time = time.time() - t0

y_pred_xgb = xgb_booster.predict(dtest)

with mlflow.start_run(run_name='xgb_v2_no_charge_colab'):
    mlflow.log_params({
        **XGB_PARAMS, 'n_rounds': 1000, 'early_stopping': 50,
        'best_iteration': xgb_booster.best_iteration,
        'source': 'colab', 'strategy': 'full_dmatrix', 'target_transform': 'log1p',
        'sample_frac': 1.0, 'split_strategy': 'random',
        'n_features': len(FEATURES), 'ablation_avg_submitted_charge': True,
        'version': 'v2', 'training_time_sec': int(xgb_time),
        'n_train': n_train, 'n_test': n_test,
    })
    log_metrics(y_test, y_pred_xgb, prefix='test_')
    mlflow.log_dict(xgb_booster.get_score(importance_type='gain'), 'feature_importances.json')
    mlflow.xgboost.log_model(xgb_booster, artifact_path='xgb_model')

xgb_booster.save_model(f'{ARTIFACTS}/models/xgb_v2_no_charge.json')

# Save predictions, free everything
np.save('/content/y_test.npy', y_test)
np.save('/content/y_pred_xgb.npy', y_pred_xgb)
del dtrain, dtest, xgb_booster, X_test, y_train, y_test, y_pred_xgb; gc.collect()
print('XGBoost done — predictions saved to /content/y_pred_xgb.npy')

In [ ]:
# ── Cell 5: CatBoost No-Charge ─────────────────────────────────────────────
# Reload splits from local SSD (DataFrame for CatBoost Pool)
df_train = pd.read_parquet('/content/train_split.parquet')
df_test  = pd.read_parquet('/content/test_split.parquet')
y_train = df_train[TARGET].values
y_test  = df_test[TARGET].values

CB_PARAMS = {
    'loss_function': 'RMSE', 'learning_rate': 0.05, 'depth': 6,
    'subsample': 0.8, 'rsm': 0.8, 'task_type': 'GPU',
    'random_seed': 42, 'verbose': 100, 'allow_writing_files': False,
    'iterations': 1000,
}

cat_idx = get_cat_indices(FEATURES)
pool_train = make_pool(df_train, y_train, FEATURES, cat_idx)
pool_test  = make_pool(df_test,  y_test,  FEATURES, cat_idx)
del df_train, df_test; gc.collect()

print('Training CatBoost (no charge)...')
t0 = time.time()
cb_model = CatBoostRegressor(**CB_PARAMS)
cb_model.fit(pool_train, eval_set=pool_test, early_stopping_rounds=50, use_best_model=True)
cb_time = time.time() - t0

y_pred_cb = cb_model.predict(pool_test)

with mlflow.start_run(run_name='catboost_v2_no_charge_colab'):
    mlflow.log_params({
        **{k: v for k, v in CB_PARAMS.items() if k != 'verbose'},
        'best_iteration': cb_model.best_iteration_,
        'source': 'colab', 'strategy': 'full_pool', 'target_transform': 'log1p',
        'sample_frac': 1.0, 'split_strategy': 'random',
        'n_features': len(FEATURES), 'n_cat_features': len(cat_idx),
        'ablation_avg_submitted_charge': True,
        'version': 'v2', 'training_time_sec': int(cb_time),
        'n_train': n_train, 'n_test': n_test,
    })
    log_metrics(y_test, y_pred_cb, prefix='test_')
    mlflow.log_dict(dict(zip(FEATURES, cb_model.get_feature_importance())), 'feature_importances.json')
    mlflow.catboost.log_model(cb_model, artifact_path='catboost_model')

cb_model.save_model(f'{ARTIFACTS}/models/catboost_v2_no_charge.cbm')

# Save predictions, free everything
np.save('/content/y_pred_cb.npy', y_pred_cb)
del pool_train, pool_test, cb_model, y_train, y_test, y_pred_cb; gc.collect()
print('CatBoost done — predictions saved to /content/y_pred_cb.npy')

In [ ]:
# ── Cell 6: LightGBM No-Charge ─────────────────────────────────────────────
import lightgbm as lgb

# Reload splits from local SSD
df_train = pd.read_parquet('/content/train_split.parquet')
df_test  = pd.read_parquet('/content/test_split.parquet')
X_train = df_train[FEATURES].astype('float32').values
y_train = df_train[TARGET].values
X_test  = df_test[FEATURES].astype('float32').values
y_test  = df_test[TARGET].values
del df_train, df_test; gc.collect()

LGB_PARAMS = {
    'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.05,
    'num_leaves': 63, 'max_depth': -1, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'min_child_samples': 20,
    'device': 'gpu', 'seed': 42, 'verbose': -1,
    'boosting_type': 'gbdt', 'data_sample_strategy': 'goss',
}

cat_cols = [FEATURES[i] for i in get_cat_indices(FEATURES)]
ds_train = lgb.Dataset(X_train, label=y_train, feature_name=FEATURES,
                       categorical_feature=cat_cols, free_raw_data=True)
ds_test  = lgb.Dataset(X_test, label=y_test, feature_name=FEATURES,
                       categorical_feature=cat_cols, reference=ds_train, free_raw_data=True)

print('Training LightGBM (no charge)...')
t0 = time.time()
lgb_booster = lgb.train(
    LGB_PARAMS, ds_train, num_boost_round=1000,
    valid_sets=[ds_test], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
lgb_time = time.time() - t0

y_pred_lgb = lgb_booster.predict(X_test)

with mlflow.start_run(run_name='lgbm_v2_no_charge_colab'):
    mlflow.log_params({
        **{k: v for k, v in LGB_PARAMS.items() if k != 'verbose'},
        'n_rounds': 1000, 'early_stopping': 50,
        'best_iteration': lgb_booster.best_iteration,
        'source': 'colab', 'strategy': 'full_dataset', 'target_transform': 'log1p',
        'sample_frac': 1.0, 'split_strategy': 'random',
        'n_features': len(FEATURES), 'n_cat_features': len(cat_cols),
        'ablation_avg_submitted_charge': True,
        'version': 'v2', 'training_time_sec': int(lgb_time),
        'n_train': n_train, 'n_test': n_test,
    })
    log_metrics(y_test, y_pred_lgb, prefix='test_')
    mlflow.log_dict(dict(zip(FEATURES, lgb_booster.feature_importance(importance_type='gain'))), 'feature_importances.json')
    mlflow.lightgbm.log_model(lgb_booster, artifact_path='lgbm_model')

lgb_booster.save_model(f'{ARTIFACTS}/models/lgbm_v2_no_charge.txt')

# Save predictions, free everything
np.save('/content/y_pred_lgb.npy', y_pred_lgb)
del ds_train, ds_test, lgb_booster, X_train, X_test, y_train, y_test, y_pred_lgb; gc.collect()
print('LightGBM done — predictions saved to /content/y_pred_lgb.npy')

In [ ]:
# ── Cell 7: Ablation Summary ───────────────────────────────────────────────
# Reload predictions from local SSD
y_test     = np.load('/content/y_test.npy')
y_pred_xgb = np.load('/content/y_pred_xgb.npy')
y_pred_cb  = np.load('/content/y_pred_cb.npy')
y_pred_lgb = np.load('/content/y_pred_lgb.npy')

print('\n' + '='*70)
print('V2_02 NO-CHARGE ABLATION COMPLETE')
print('='*70)
print(f'Features: {len(FEATURES)} (WITHOUT Avg_Sbmtd_Chrg)')
print()

for name, y_pred in [('XGBoost', y_pred_xgb), ('CatBoost', y_pred_cb), ('LightGBM', y_pred_lgb)]:
    yt = np.expm1(y_test)
    yp = np.expm1(y_pred)
    mae  = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    r2   = r2_score(yt, yp)
    print(f'{name:12s}  MAE=${mae:8.2f}  RMSE=${rmse:8.2f}  R²={r2:.4f}')

print()
print('Compare with V2_01 (with charge) to measure Avg_Sbmtd_Chrg dependency.')
print('The R² delta = V2_01_R² - V2_02_R² quantifies how much the model')
print('relies on knowing the billed charge vs. genuinely predicting cost.')